# Fine-tuned DINOv2-S/14

This self-contained notebook loads the pretrained `vit_small_patch14_dinov2.lvd142m` model and fine-tunes **all model parameters** for 20 epochs. It uses the same manifest-defined split as the other experiments: 176 training images and 45 validation images.

This run is substantially slower than the pretrained feature-extractor experiment because every image passes through the full model during every epoch.

In [ ]:
import csv
import json
import os
import random
import sys
import warnings
from pathlib import Path

project_candidates = [
    Path.home() / 'Documents' / 'Paper replication',
    Path.home() / 'Desktop' / 'Paper replication',
]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / 'data' / 'common_split_manifest.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the Paper replication project in Documents or Desktop.')

MODEL_FOLDER = PROJECT_ROOT / 'model_reproductions' / 'no_benefit_dinov2_fine_tuned'
MANIFEST = PROJECT_ROOT / 'data' / 'common_split_manifest.csv'
OUTPUT_FOLDER = MODEL_FOLDER / 'results_fine_tuned'
MODEL_NAME = 'vit_small_patch14_dinov2.lvd142m'
CLASS_NAMES = ['Low', 'High']

MODEL_CACHE = (
    Path.home() / '.cache' / 'huggingface' / 'hub'
    / 'models--timm--vit_small_patch14_dinov2.lvd142m'
)
if MODEL_CACHE.is_dir():
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL.*')
warnings.filterwarnings('ignore', message='IProgress not found.*')

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def choose_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

In [ ]:
def read_split():
    with MANIFEST.open(newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))
    train, validation = [], []
    for row in rows:
        folder = 'combined_high_bw' if int(row['label']) else 'combined_low_bw'
        path = MANIFEST.parent / folder / row['file']
        if not path.is_file():
            raise FileNotFoundError(path)
        sample = (path, int(row['label']), row['file'])
        (validation if row['split'] == 'validation' else train).append(sample)
    assert len(train) == 176 and len(validation) == 45
    assert not ({x[2] for x in train} & {x[2] for x in validation})
    return train, validation

class ImageDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label, filename = self.samples[index]
        image = Image.open(path).convert('L').convert('RGB')
        return self.transform(image), label, filename

def make_loaders(batch_size=8):
    train, validation = read_split()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])
    generator = torch.Generator().manual_seed(42)
    train_loader = DataLoader(
        ImageDataset(train, transform), batch_size=batch_size, shuffle=True,
        num_workers=0, generator=generator,
    )
    validation_loader = DataLoader(
        ImageDataset(validation, transform), batch_size=batch_size,
        shuffle=False, num_workers=0,
    )
    return train_loader, validation_loader

In [ ]:
def run_epoch(model, loader, loss_function, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct = 0.0, 0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            scores = model(images)
            loss = loss_function(scores, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += int((scores.argmax(1) == labels).sum())
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    actual, predicted, probabilities, filenames = [], [], [], []
    with torch.inference_mode():
        for images, labels, names in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            probs = torch.softmax(scores, dim=1)
            total_loss += loss_function(scores, labels).item() * len(labels)
            actual.extend(labels.cpu().tolist())
            predicted.extend(scores.argmax(1).cpu().tolist())
            probabilities.extend(probs.cpu().tolist())
            filenames.extend(names)
    report = classification_report(
        actual, predicted, labels=[0, 1], target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    correct = sum(a == b for a, b in zip(actual, predicted))
    metrics = {
        'validation_loss': total_loss / len(loader.dataset),
        'validation_accuracy': correct / len(loader.dataset),
        'correct_predictions': correct,
        'incorrect_predictions': len(loader.dataset) - correct,
        'prediction_count': len(loader.dataset),
        'confusion_matrix': confusion_matrix(actual, predicted, labels=[0, 1]).tolist(),
        'Low': report['Low'], 'High': report['High'],
        'macro_f1': report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score'],
    }
    return metrics, list(zip(filenames, actual, predicted, probabilities))

def save_evaluation(name, model, loader, loss_function, device, output_dir):
    metrics, predictions = evaluate(model, loader, loss_function, device)
    (output_dir / f'{name}_metrics.json').write_text(json.dumps(metrics, indent=2))
    with (output_dir / f'{name}_predictions.csv').open('w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['file', 'true_label', 'predicted_label', 'probability_low', 'probability_high', 'correct'])
        for filename, true, prediction, probs in predictions:
            writer.writerow([filename, true, prediction, probs[0], probs[1], true == prediction])
    return metrics

In [ ]:
def fine_tune_dinov2(model, output_dir, epochs=20, learning_rate=1e-5):
    set_seed(42)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    device = choose_device()
    for parameter in model.parameters():
        parameter.requires_grad = True
    model.to(device)
    train_loader, validation_loader = make_loaders(batch_size=8)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    history = []
    best_loss, best_accuracy, best_accuracy_loss = float('inf'), -1.0, float('inf')
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_epoch(
            model, train_loader, loss_function, device, optimizer
        )
        validation_loss, validation_accuracy = run_epoch(
            model, validation_loader, loss_function, device
        )
        row = {
            'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_accuracy,
            'validation_loss': validation_loss, 'validation_accuracy': validation_accuracy,
        }
        history.append(row)
        print(json.dumps(row))
        if validation_loss < best_loss:
            best_loss = validation_loss
            torch.save(model.state_dict(), output_dir / 'best_validation_loss.pth')
        if validation_accuracy > best_accuracy or (
            validation_accuracy == best_accuracy and validation_loss < best_accuracy_loss
        ):
            best_accuracy, best_accuracy_loss = validation_accuracy, validation_loss
            torch.save(model.state_dict(), output_dir / 'best_validation_accuracy.pth')
    torch.save(model.state_dict(), output_dir / 'final_epoch.pth')

    accuracy_rows = [
        {'epoch': row['epoch'], 'training_accuracy': row['train_accuracy'],
         'validation_accuracy': row['validation_accuracy']}
        for row in history
    ]
    with (output_dir / 'accuracy_by_epoch.csv').open('w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(accuracy_rows[0]))
        writer.writeheader()
        writer.writerows(accuracy_rows)

    epoch_numbers = [row['epoch'] for row in history]
    training_accuracy = [row['train_accuracy'] for row in history]
    validation_accuracy = [row['validation_accuracy'] for row in history]
    figure, axis = plt.subplots(figsize=(9, 5.5))
    axis.plot(epoch_numbers, training_accuracy, 'o-', linewidth=2, label='Training accuracy')
    axis.plot(epoch_numbers, validation_accuracy, 'o-', linewidth=2, label='Validation accuracy')
    for values, label in ((training_accuracy, 'Training'), (validation_accuracy, 'Validation')):
        best_index = int(np.argmax(values))
        axis.scatter(epoch_numbers[best_index], values[best_index], s=80, zorder=5)
        axis.annotate(
            f'{label} max: {values[best_index]:.1%}',
            (epoch_numbers[best_index], values[best_index]),
            xytext=(0, 12), textcoords='offset points', ha='center',
        )
    axis.set(xlabel='Epoch', ylabel='Accuracy', title='Fine-tuned DINOv2 accuracy',
             xlim=(1, epochs), ylim=(0, 1.08))
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    figure.savefig(output_dir / 'accuracy_graph.png', dpi=220, bbox_inches='tight')
    if 'ipykernel' in sys.modules:
        plt.show()
    plt.close(figure)

    summary = {}
    for name, checkpoint in (
        ('final', 'final_epoch.pth'),
        ('best_loss', 'best_validation_loss.pth'),
        ('best_accuracy', 'best_validation_accuracy.pth'),
    ):
        model.load_state_dict(torch.load(output_dir / checkpoint, map_location=device, weights_only=True))
        summary[name] = save_evaluation(
            name, model, validation_loader, loss_function, device, output_dir
        )

    result_rows = [
        {
            'checkpoint': name,
            'validation_accuracy': metrics['validation_accuracy'],
            'validation_loss': metrics['validation_loss'],
            'correct_predictions': metrics['correct_predictions'],
            'incorrect_predictions': metrics['incorrect_predictions'],
            'macro_f1': metrics['macro_f1'],
            'weighted_f1': metrics['weighted_f1'],
        }
        for name, metrics in summary.items()
    ]
    with (output_dir / 'validation_results.csv').open('w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(result_rows[0]))
        writer.writeheader()
        writer.writerows(result_rows)

    configuration = {
        'model': MODEL_NAME, 'method': 'all-parameter fine-tuning',
        'epochs': epochs, 'learning_rate': learning_rate, 'weight_decay': 1e-4,
        'seed': 42, 'training_images': 176, 'validation_images': 45, 'test_images': 0,
        'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'total_parameters': sum(p.numel() for p in model.parameters()), 'device': str(device),
    }
    (output_dir / 'run_configuration.json').write_text(json.dumps(configuration, indent=2))
    (output_dir / 'result_summary.json').write_text(json.dumps(summary, indent=2))
    print('\nValidation results:')
    for row in result_rows:
        print(f"{row['checkpoint']:>13}: {row['validation_accuracy']:.2%} "
              f"({row['correct_predictions']}/45 correct)")
    return summary

In [ ]:
set_seed(42)
model = timm.create_model(
    MODEL_NAME, pretrained=True, num_classes=2, img_size=224
)
print(f'Loaded pretrained DINOv2-S/14 ({sum(p.numel() for p in model.parameters()):,} parameters)')
print('All model parameters will be fine-tuned.')

results = fine_tune_dinov2(
    model=model, output_dir=OUTPUT_FOLDER, epochs=20, learning_rate=1e-5
)

The `results_fine_tuned` folder contains the epoch accuracy CSV, checkpoint summary CSV, per-image validation predictions, accuracy graph, model checkpoints, and run configuration. There is no independent test split, so all held-out scores are correctly reported as **validation accuracy**.